# Part B · Token-Level EDA

## Pipeline Position

| Part | Notebook | GPU Required | Input | Output |
|------|----------|-------------|-------|--------|
| A | part_a_translation_evaluation | Yes (T4) | FLoRes-200 dataset | sacrebleu_results.csv |
| **B** | **part_b_token_eda** | **No (CPU)** | **sacrebleu_results.csv** | **token_counts.csv, engineered_features.csv** |
| C | part_c_indic_token_analysis | No (CPU) | token_counts.csv | Vocabulary coverage charts |

## What This Notebook Measures

| Metric | Definition | Why It Matters |
|--------|-----------|----------------|
| `expansion_ratio` | Target tokens ÷ Source tokens | Measures verbosity of the tokeniser for Tamil |
| `avg_word_length` | Tamil chars ÷ Target token count | Higher = fewer fragmented subwords |
| `subword_fragmentation` | 1 ÷ avg_word_length | Proxy for aggressive subword splitting |
| `unknown_token_rate` | % tokens mapped to UNK | Measures out-of-vocabulary exposure |

## Why Token Analysis Matters Beyond BLEU

BLEU measures surface-level n-gram overlap — it does not reveal **how** a model internally represents Tamil text. Two models with similar BLEU scores can have drastically different tokenisation behaviours:

- A model may fragment every Tamil word into 4–6 tiny subwords, bloating sequence length and increasing attention memory cost
- Another model may encode the same sentence in half the tokens, with richer information per token

These tokenisation differences affect:
- **Transformer attention memory** — O(n²) cost scales with sequence length squared
- **Information density** — longer tokens carry more morphological signal
- **Agglutination handling** — Tamil verbs like வந்திருக்கிறான் ("he has come") encode 4+ morphemes in one word; a poor tokeniser cannot represent this as a single meaningful unit

**Data source:** `../part_a_batch_translation/sacrebleu_results.csv` — 100 FLoRes-200 English–Tamil sentence pairs with translations from all 5 models.

## Model Reference

| Model | Size | Tokeniser Type | Tamil Focus |
|-------|------|---------------|-------------|
| IndicTrans2 | 1B | SentencePiece (22 Indic langs) | High |
| NLLB-200 | 600M | SentencePiece (200 langs) | Moderate |
| mT5 | 580M | SentencePiece (101 langs) | Low (tokeniser analysis only) |
| Helsinki MarianMT | 74M | SentencePiece (EN→Dravidian: ta/kn/ml/te) | High |
| MADLAD-400 | 3B | SentencePiece (400 langs) | Low (extreme dilution) |

## Section 1 · Environment Setup

**CPU-only notebook** — no GPU required for tokenisation. All five tokenisers run as CPU-bound string operations; no matrix multiplications are needed to tokenise text.

**Why a consistent colour palette?** All three notebooks (A, B, C) use the same `MODEL_COLORS` dictionary keyed by model name. This ensures that IndicTrans2 is always `#2E86AB` (blue), NLLB-200 is always `#A23B72` (purple), and so on across every chart in the project. A reviewer comparing plots across notebooks can identify models at a glance without reading the legend each time.

In [ ]:
# ── Cell 0 · Runtime check + global visual theme ──────────────────────────────
import subprocess, sys, os, gc, torch, warnings
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
import pandas as pd
from IPython.display import display, HTML
warnings.filterwarnings("ignore")

print("Python :", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs("plots", exist_ok=True)

PALETTE = ["#2E86AB", "#A23B72", "#F18F01", "#C73E1D", "#3B1F2B"]
MODEL_COLORS = {
    "IndicTrans2" : "#2E86AB",
    "NLLB-200"    : "#A23B72",
    "mT5"         : "#F18F01",
    "Helsinki"    : "#C73E1D",
    "MADLAD"      : "#3B1F2B",
}
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 150, "figure.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "font.family": "DejaVu Sans",
})
print("\u2713 Global theme applied")

## Why mT5 Is Included Here (but Not in BLEU Evaluation)

mT5 is a **masked seq2seq pre-training model** — it was trained to reconstruct corrupted input spans, not to translate from English to Tamil. Its raw decoded outputs are not meaningful translations and would score near zero on sacreBLEU.

However, its **SentencePiece tokeniser** (trained on mC4, a 101-language corpus) is architecturally significant as a baseline:

- It has one of the largest vocabularies (~250K tokens) yet Tamil receives a very limited vocabulary budget due to English-heavy training data
- Comparing mT5 tokenisation against IndicTrans2 (Indic-specialised, ~64K vocab) directly illustrates the **vocabulary dilution problem** in massively multilingual models: more languages in one tokeniser means fewer tokens available per language

**Design decision:** mT5 is included in all tokenisation charts (Parts B and C) but excluded from BLEU scoring (Part A). This is intentional and explicitly flagged in every relevant visualisation.

## Section 2 · Loading Tokenisers

We load only the **tokeniser weights** — not the full model — using `AutoTokenizer.from_pretrained()`. This is fast (seconds per tokeniser) and requires no GPU.

| Model | Tokeniser Type | Expected Vocab Size | Tamil Script Coverage |
|-------|---------------|---------------------|----------------------|
| IndicTrans2 | SentencePiece (22 Indic scripts) | ~64,000 | High — dedicated Indic vocabulary allocation |
| NLLB-200 | SentencePiece (200 languages) | ~256,000 | Moderate — large vocab but diluted across 200 languages |
| mT5 | SentencePiece (mC4 corpus, 101 langs) | ~250,000 | Low — English-heavy training data |
| Helsinki | SentencePiece (EN→Dravidian multilingual: ta/kn/ml/te) | ~65,000 | High — vocabulary focused on 4 Dravidian scripts |
| MADLAD-400 | SentencePiece (400 languages) | ~256,000 | Low — extreme multilingual dilution |

**Key insight:** A larger vocabulary does not mean better Tamil coverage. NLLB-200 has 256K tokens yet Tamil occupies only a fraction of that space. IndicTrans2 with 64K tokens focused on 22 Indic scripts achieves denser, more meaningful Tamil subword representation — the vocabulary is compact but well-allocated.

In [ ]:
# ── Cell 2 · Load tokenizers (no model weights) ───────────────────────────────
from transformers import AutoTokenizer

TOKENIZER_IDS = {
    "IndicTrans2": ("ai4bharat/indictrans2-en-indic-1B", {"trust_remote_code": True}),
    "NLLB-200":    ("facebook/nllb-200-distilled-600M", {}),
    "mT5":         ("google/mt5-base", {}),
    "Helsinki":    ("Helsinki-NLP/opus-mt-en-dra", {}),
    "MADLAD":      ("google/madlad400-3b-mt", {}),
}

tokenizers = {}
for name, (model_id, kwargs) in TOKENIZER_IDS.items():
    print(f"Loading tokenizer: {name}")
    tokenizers[name] = AutoTokenizer.from_pretrained(model_id, **kwargs)
    print(f"  ✓ Vocab size: {tokenizers[name].vocab_size:,}")

## Section 3 · Data Handoff from Part A

Part B reads the CSV saved by Part A. Expected schema:

| Column | Description |
|--------|-------------|
| `source_english` or `english` | Original English sentence from FLoRes-200 devtest |
| `reference_tamil` | Human-verified Tamil reference translation |
| `pred_IndicTrans2` | IndicTrans2 model output |
| `pred_NLLB-200` | NLLB-200-distilled-600M output |
| `pred_Helsinki` | Helsinki MarianMT en-dra output |
| `pred_MADLAD` | MADLAD-400-3b-mt output |
| `pred_mT5` | mT5-base raw decoded output (not a translation) |

**Column name handling:** The blueprint specifies `english` but our Part A saves the column as `source_english`. The load cell handles both variants with a conditional check — if Part A is re-run with the blueprint column name, Part B continues to work without modification.

**mT5 note:** `pred_mT5` contains text decoded from the mT5 model without any translation fine-tuning. The actual decoded strings are not meaningful Tamil translations, but we tokenise them through the mT5 tokeniser to measure how mT5 handles Tamil text regardless of decoding quality.

In [ ]:
# ── Cell 3 · Load Part A translations ─────────────────────────────────────────
df_parta = pd.read_csv("../part_a_batch_translation/sacrebleu_results.csv")
print(f"\u2713 Loaded Part A results: {len(df_parta)} rows")
print(f"  Columns: {list(df_parta.columns)}")

# Handle column name difference between blueprint ("english") and our Part A ("source_english")
eng_col = "english" if "english" in df_parta.columns else "source_english"
sentences_en = df_parta[eng_col].tolist()

model_outputs = {
    "IndicTrans2": df_parta["pred_IndicTrans2"].tolist(),
    "NLLB-200":   df_parta["pred_NLLB-200"].tolist(),
    "Helsinki":   df_parta["pred_Helsinki"].tolist(),
    "MADLAD":     df_parta["pred_MADLAD"].tolist(),
    "mT5":        df_parta["pred_mT5"].tolist(),
}
print("\u2713 All 5 model outputs loaded (including mT5 for tokenization analysis)")

## Section 4 · Token Metrics Computation

Each of the 100 English sentences and its 5 corresponding Tamil translations is passed through each model's tokeniser. We compute 4 core metrics per sentence per model (500 total measurements):

| Metric | Formula | Interpretation |
|--------|---------|---------------|
| `expansion_ratio` | target_tokens ÷ source_tokens | >1.0 = Tamil is more verbose than English in this tokeniser |
| `avg_word_length` | tamil_chars ÷ target_tokens | Higher = larger, more meaningful subword units |
| `subword_fragmentation` | 1 ÷ avg_word_length | Higher = finer-grained splitting; harder for the model to decode |
| `unknown_token_rate` | (UNK_count ÷ target_tokens) × 100 | Higher = more out-of-vocabulary Tamil words |

**Why `encode()` instead of `tokenize()`?**
We use `tokenizer.encode()` (returns integer token IDs) rather than `tokenizer.tokenize()` (returns string tokens) specifically to detect UNK tokens. An UNK is identified by comparing each ID to `tokenizer.unk_token_id`. String-based `tokenize()` returns `<unk>` as a plain string, which cannot be reliably distinguished from a genuine vocabulary token also named `<unk>` — a subtle but real edge case in some tokeniser implementations.

**Tamil character counting** strips whitespace before dividing by token count. Tamil is written with no spaces between characters within a word, so the raw character count directly measures how many Tamil script code points each token encodes on average. A higher value means each token carries more linguistic content.

In [ ]:
# ── Cell 4 · Compute token metrics ────────────────────────────────────────────
def clean_tokens(token_list):
    cleaned = []
    for tok in token_list:
        tok = tok.replace("\u2581", "").replace("##", "").strip()
        if tok:
            cleaned.append(tok)
    return cleaned


def compute_token_metrics(text_en, text_ta, tokenizer):
    src_tokens = tokenizer.encode(text_en, add_special_tokens=False)
    tgt_tokens = tokenizer.encode(text_ta, add_special_tokens=False)
    src_count = max(len(src_tokens), 1)
    tgt_count = max(len(tgt_tokens), 1)
    unk_id = tokenizer.unk_token_id
    unk_rate = (tgt_tokens.count(unk_id) / tgt_count * 100) if unk_id else 0.0
    tamil_chars = len(str(text_ta).replace(" ", ""))
    avg_chars_per_tok = tamil_chars / tgt_count
    subword_frag = round(1 / avg_chars_per_tok, 4) if avg_chars_per_tok > 0 else 0
    return {
        "source_token_count": src_count,
        "target_token_count": tgt_count,
        "expansion_ratio": round(tgt_count / src_count, 3),
        "avg_word_length": round(avg_chars_per_tok, 3),
        "subword_fragmentation": subword_frag,
        "unknown_token_rate": round(unk_rate, 3),
    }


records = []
for model_name, translations in model_outputs.items():
    tok = tokenizers[model_name]
    for idx, (en, ta) in enumerate(zip(sentences_en, translations)):
        metrics = compute_token_metrics(en, str(ta), tok)
        records.append({"model": model_name, "sentence_id": idx, "english": en, "tamil": ta, **metrics})

token_df = pd.DataFrame(records)
token_df.to_csv("token_counts.csv", index=False)
print(f"\u2713 Token metrics computed: {len(token_df)} rows")
print(token_df.groupby("model")["expansion_ratio"].mean().round(3))

## Section 5 · Feature Engineering

Three derived features augment the raw metrics to support richer downstream analysis and visualisation:

| Derived Feature | Formula | Purpose |
|-----------------|---------|---------|
| `log_expansion` | `log1p(expansion_ratio)` | Log-transforms expansion ratio to reduce skew from agglutinated-verb outliers |
| `efficiency_score` | `avg_word_length ÷ expansion_ratio` | Composite quality index: rewards long, information-dense tokens without bloating sequence length |
| `fragmentation_class` | `pd.cut(subword_fragmentation, bins=[0, 0.2, 0.5, 1.0, ∞])` | Categorical label (Low / Medium / High / Very High) for group-level filtering |

**Why log-transform expansion ratio?**
Tamil verb phrases with heavy suffixation can produce expansion ratios of 5–10× for a single English phrase (e.g., "they have been working" → one agglutinated Tamil word → fragmented into 8–12 subwords). Without the log transform, these outliers compress the visual range for the 90% of sentences with ratios between 1.0 and 3.0, making model differences invisible.

**Why `efficiency_score`?**
No single metric captures tokeniser quality in isolation. A model could achieve a low expansion ratio by mapping Tamil to UNK tokens — technically reducing sequence length but destroying meaning entirely. `efficiency_score` penalises this: it rewards high `avg_word_length` only when expansion remains reasonable. A tokeniser that achieves both scores high; one that games expansion at the cost of UNK tokens scores low.

In [ ]:
# ── Cell 5 · Feature engineering ──────────────────────────────────────────────
token_df["log_expansion"] = np.log1p(token_df["expansion_ratio"])
token_df["efficiency_score"] = token_df["avg_word_length"] / token_df["expansion_ratio"]
token_df["fragmentation_class"] = pd.cut(
    token_df["subword_fragmentation"],
    bins=[0, 0.2, 0.5, 1.0, float("inf")],
    labels=["Low", "Medium", "High", "Very High"],
)
token_df.to_csv("engineered_features.csv", index=False)
print("\u2713 Engineered features saved")

## VIZ B1 · Radar Chart — Four-Metric Tokeniser Comparison

**What is being shown:** Each model is drawn as a filled polygon on a 4-axis polar chart. All axes are normalised to [0, 1] so metrics with different units and scales can be compared on the same chart.

**Normalisation direction — outer edge always means better:**
- `avg_word_length` — higher is better → plotted directly (outer = high value = better)
- `expansion_ratio`, `subword_fragmentation`, `unknown_token_rate` — lower is better → each is inverted `(1 − normalised)` before plotting so outer edge still means best performance

**How to read it:**
- A model that fills **more total area** has the strongest all-round tokeniser profile for Tamil
- A model with an **irregular polygon** performs well on some metrics but trades off poorly on others
- **Overlapping polygons** indicate models with similar tokeniser behaviour across multiple dimensions

**Why radar over four separate bar charts?**
Bar charts would require 4 separate subplots and multiple legend lookups across them. The radar chart simultaneously shows: (1) absolute normalised performance on each axis, (2) trade-offs between metrics (e.g., great expansion ratio but high UNK rate), and (3) the overall balance of each tokeniser across all four dimensions in a single glance.

In [ ]:
# ── Cell 6 · VIZ B1 · Radar chart (4 tokenizer metrics) ──────────────────────
METRICS = ["expansion_ratio", "avg_word_length", "subword_fragmentation", "unknown_token_rate"]
LABELS  = ["Expansion\nRatio", "Avg Word\nLength", "Subword\nFragmentation", "Unknown\nToken Rate"]
HIGHER_IS_BETTER = [False, True, False, False]

summary = token_df.groupby("model")[METRICS].mean()
norm = summary.copy()
for col, higher in zip(METRICS, HIGHER_IS_BETTER):
    mn, mx = summary[col].min(), summary[col].max()
    if mx == mn:
        norm[col] = 0.5
    elif higher:
        norm[col] = (summary[col] - mn) / (mx - mn)
    else:
        norm[col] = 1 - (summary[col] - mn) / (mx - mn)

N      = len(METRICS)
angles = [n / N * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
for model_name, color in MODEL_COLORS.items():
    if model_name not in norm.index:
        continue
    values  = norm.loc[model_name].values.flatten().tolist()
    values += values[:1]
    ax.plot(angles, values, "o-", linewidth=2, label=model_name, color=color)
    ax.fill(angles, values, alpha=0.08, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(LABELS, size=10, fontweight="bold")
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(["25%", "50%", "75%", "100%"], size=8, color="grey")
ax.grid(color="grey", linestyle="--", linewidth=0.5, alpha=0.5)
ax.set_title(
    "Model Tokenizer Comparison\n(Outer = Better on that metric)",
    size=15, fontweight="bold", pad=25,
)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=11)
plt.tight_layout()
plt.savefig("plots/partb_radar_chart.png", bbox_inches="tight", dpi=150)
plt.show()

## VIZ B2 · Bubble Chart — Token Expansion per Sentence

**What this chart shows:** Each bubble is one sentence (100 per model). The X-axis encodes source token count (English), the Y-axis encodes target token count (Tamil), and **bubble size is proportional to `expansion_ratio`** for that sentence. All five models are overlaid using the shared MODEL_COLORS palette.

**The reference diagonal (y = x, dashed):** A model that expands source tokens 1:1 sits exactly on this line. Every Tamil model scatters *above* it because Tamil morphology requires more tokens than English — the question is *how far above* and *how consistently across sentence lengths*.

| Region | Interpretation |
|--------|----------------|
| Near diagonal | Near-parity expansion; efficient subword segmentation |
| Far above diagonal | High expansion; model fragments Tamil words into many pieces |
| Large bubble far above diagonal | Worst-case: long English source AND high fragmentation combined |
| Vertical scatter within one model | Inconsistent behaviour; some sentence types are handled better than others |

**Model-specific patterns to look for:**
- **IndicTrans2** and **Helsinki** cluster closer to the diagonal; IndicTrans2 benefits from a dedicated 22-Indic-script vocabulary (~64K tokens)
- **MADLAD-400** and **NLLB-200** show wider vertical scatter; their massively multilingual vocabularies dilute Tamil coverage, producing higher fragmentation on morphologically complex sentences
- **mT5** is plotted as a tokenisation reference; its bubbles reflect raw Tamil subword behaviour unrelated to translation quality

**Design choice — bubble chart over a line plot:** Aggregating across sentences hides the sentence-level variance that reveals fragmentation hot-spots. The bubble chart preserves all 100 data points per model while the size encoding adds a third dimension without a separate subplot.

In [ ]:
# ── Cell 7 · VIZ B2 · Bubble chart (token expansion) ─────────────────────────
fig, ax = plt.subplots(figsize=(12, 7))
for model_name, color in MODEL_COLORS.items():
    sub = token_df[token_df["model"] == model_name]
    ax.scatter(
        sub["source_token_count"], sub["target_token_count"],
        s=sub["expansion_ratio"] * 120, c=color, alpha=0.65,
        edgecolors="white", linewidths=0.8, label=model_name,
    )

xvals = np.linspace(
    token_df["source_token_count"].min(),
    token_df["source_token_count"].max(),
    100,
)
ax.plot(xvals, xvals, "k--", linewidth=1, alpha=0.4, label="Ratio = 1.0")
ax.set_xlabel("Source Token Count (English)", fontsize=13)
ax.set_ylabel("Target Token Count (Tamil)", fontsize=13)
ax.set_title(
    "Token Expansion Bubble Chart\n(Bubble size \u221d expansion ratio)",
    fontsize=15, fontweight="bold",
)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig("plots/partb_bubble_chart.png", bbox_inches="tight", dpi=150)
plt.show()

## VIZ B3 · Violin Plot — Expansion Ratio Distribution

**What this chart shows:** Each violin represents the full distribution of `expansion_ratio` across 100 sentences for one model. The inner box shows the interquartile range (25th–75th percentile) and median. The outer shape is a kernel density estimate — **wider sections indicate higher data density** at that expansion ratio value.

**The dashed line at 1.0:** Marks the theoretical point of parity (Tamil and English tokenise to the same number of tokens). All models should sit above this line for Tamil, which is expected given Tamil's agglutinative morphology. A model whose median sits *close* to 1.0 has an unusually efficient tokeniser.

**How to read the violin shape:**
- **Wide belly near 1.0–2.0** = most sentences are tokenised efficiently; expansion is moderate and predictable
- **Long upper tail** = a subset of sentences (typically those with complex verb phrases or compound nouns) cause dramatically higher expansion
- **Bimodal shape** (two distinct bulges) = the model handles simple and complex sentences very differently, suggesting vocabulary gaps for specific Tamil morphological patterns

**Why violin over a plain boxplot?**
A boxplot shows only the quartiles and outliers — it cannot reveal whether the distribution is unimodal or bimodal. For Tamil tokenisation, bimodality is diagnostically important: a bimodal violin suggests the model handles simple nominal phrases well but fails on agglutinated verb clusters. A boxplot would mask this entirely.

In [ ]:
# ── Cell 8 · VIZ B3 · Violin plot (expansion ratio distribution) ──────────────
fig, ax = plt.subplots(figsize=(12, 6))
sns.violinplot(
    data=token_df, x="model", y="expansion_ratio",
    order=list(MODEL_COLORS.keys()),
    palette=MODEL_COLORS, inner="box", linewidth=1.5, ax=ax,
)
ax.axhline(1.0, color="black", linestyle="--", linewidth=1, alpha=0.5, label="Expansion ratio = 1.0")
ax.set_title("Token Expansion Ratio Distribution by Model", fontsize=15, fontweight="bold")
ax.set_xlabel("Model")
ax.set_ylabel("Expansion Ratio (Target Tokens / Source Tokens)")
ax.legend()
plt.tight_layout()
plt.savefig("plots/partb_violin_expansion.png", bbox_inches="tight", dpi=150)
plt.show()

## VIZ B4 · Heatmap — Model × Metric Summary

**Purpose:** An executive-summary table of all four tokenisation metrics across all five models. Each cell shows the mean value across 100 sentences.

**Colour encoding — `RdYlGn_r` (reversed Red-Yellow-Green):**
The `_r` suffix inverts the default colormap so that **red = high value, green = low value**. For three of the four metrics this maps intuitively to quality:

| Metric | Red (high) means | Green (low) means |
|--------|-----------------|-------------------|
| `expansion_ratio` | Many tokens per English word | Efficient, compact Tamil representation |
| `subword_fragmentation` | Aggressive splitting of Tamil words | Whole Tamil subwords retained |
| `unknown_token_rate` | Many out-of-vocabulary Tamil tokens | All Tamil text covered by vocabulary |
| `avg_word_length` | *(exception — higher is better here)* | Short tokens; over-segmented Tamil |

**Note on `avg_word_length`:** This is the only metric where the colour direction is inverted relative to quality — a red (high) cell for `avg_word_length` actually indicates better tokeniser behaviour. This is noted explicitly because `RdYlGn_r` is applied uniformly across all columns.

**Expected pattern:** IndicTrans2 and Helsinki should show the greenest cells on expansion and fragmentation (dedicated Tamil/Indic vocabulary). MADLAD-400 and NLLB-200 may show redder cells due to multilingual vocabulary dilution. mT5's `avg_word_length` should be notably lower than IndicTrans2, reflecting finer-grained Tamil splitting despite having a nominally large vocabulary.

In [ ]:
# ── Cell 9 · VIZ B4 · Heatmap (model × metric summary) ───────────────────────
summary_display = token_df.groupby("model")[METRICS].mean().round(3)

fig, ax = plt.subplots(figsize=(11, 5))
sns.heatmap(
    summary_display, annot=True, fmt=".2f", cmap="RdYlGn_r",
    linewidths=0.5, linecolor="white", ax=ax,
    cbar_kws={"label": "Metric Value"}, annot_kws={"size": 11, "weight": "bold"},
)
ax.set_title(
    "Model \u00d7 Metric Heatmap (avg across 100 sentences)",
    fontsize=14, fontweight="bold", pad=15,
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.tight_layout()
plt.savefig("plots/partb_heatmap.png", bbox_inches="tight", dpi=150)
plt.show()